In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)

In [2]:
# ============================================================
# 2. PROJECT PATH
# ============================================================

import sys
import importlib
from pathlib import Path

project_root = Path(r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data_pipeline
import src.events

importlib.reload(src.data_pipeline)
importlib.reload(src.events)

from src.data_pipeline import ModelDatasetBuilder
from src.events import SimpleEventDetector, CombinedEventDetector

In [3]:
# ============================================================
# 3. DATABASE PATH
# ============================================================

db_path = project_root / "NordPoool" / "data" / "thesis_database.db"

print("DB exists:", db_path.exists())
print("DB path:", db_path)

builder = ModelDatasetBuilder(db_path)

DB exists: True
DB path: C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg\NordPoool\data\thesis_database.db


In [4]:
# ============================================================
# 4. LOAD DATA
# ============================================================

df = builder.build_price_dataset(
    zones="NO1",
    start_date="2020-01-01",
    end_date="2020-12-31",
    add_time_features=True,
    lags=[1, 2, 24],
    target_horizon=1,
    include_volumes=True,
    # dropna=True Ignore rows with NaN values (due to lags)
)

df.head()

,price_id,zone_id,delivery_day,hour,price_value,buy_volume_value,sell_volume_value,year,month,day,day_of_week,price_value_lag_1,price_value_lag_2,price_value_lag_24,target
datetime,,,,,,,,,,,,,,,
2020-01-01 00:00:00,32,12,2020-01-01,0,31.77,4091.8,1819.6,2020,1,1,2,NaN,NaN,NaN,31.57
2020-01-01 01:00:00,52,12,2020-01-01,1,31.57,4021.3,1826.2,2020,1,1,2,31.77,NaN,NaN,31.28
2020-01-01 02:00:00,72,12,2020-01-01,2,31.28,3975.7,1836.8,2020,1,1,2,31.57,31.77,NaN,30.72
2020-01-01 03:00:00,92,12,2020-01-01,3,30.72,3993.6,1841.5,2020,1,1,2,31.28,31.57,NaN,30.27
2020-01-01 04:00:00,112,12,2020-01-01,4,30.27,4041.5,1798.0,2020,1,1,2,30.72,31.28,NaN,30.17


In [5]:
# ============================================================
# 5. DETECT PRICE EVENTS
# ============================================================

detector = SimpleEventDetector()
combined_detector = CombinedEventDetector()

df_events = detector.detect_price_events(df)
df_events = detector.detect_volume_events(df_events)
df_events = combined_detector.detect_combined_events(df_events)

print(df_events.shape)
df_events.head()

(8761, 58)


,price_id,zone_id,delivery_day,hour,price_value,buy_volume_value,sell_volume_value,year,month,day,...,sell_pressure_low_price,system_price_max,system_price_min,system_price_spread,price_separation,extreme_price_separation,system_price_median,price_deviation_from_system,zone_price_outlier,has_combined_event
datetime,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,32,12,2020-01-01,0,31.77,4091.8,1819.6,2020,1,1,...,False,31.77,31.77,0.0,False,False,31.77,0.0,False,False
2020-01-01 01:00:00,52,12,2020-01-01,1,31.57,4021.3,1826.2,2020,1,1,...,False,31.57,31.57,0.0,False,False,31.57,0.0,False,False
2020-01-01 02:00:00,72,12,2020-01-01,2,31.28,3975.7,1836.8,2020,1,1,...,False,31.28,31.28,0.0,False,False,31.28,0.0,False,False
2020-01-01 03:00:00,92,12,2020-01-01,3,30.72,3993.6,1841.5,2020,1,1,...,False,30.72,30.72,0.0,False,False,30.72,0.0,False,False
2020-01-01 04:00:00,112,12,2020-01-01,4,30.27,4041.5,1798.0,2020,1,1,...,False,30.27,30.27,0.0,False,False,30.27,0.0,False,False


In [6]:
# ============================================================
# 6. SELECT FEATURE SETS AND TARGET
# ============================================================

price_base_features = [
    "price_value",
    "year",
    "month",
    "day",
    "day_of_week",
]

lag_features = [
    "price_value_lag_1",
    "price_value_lag_2",
    "price_value_lag_24",
]

volume_base_features = [
    "buy_volume_value",
    "sell_volume_value",
]

price_event_features = [
    "low_price",
    "high_price",
    "price_spike",
    "extreme_price",
    "rapid_price_change",
    "price_ramp_up",
    "price_ramp_down",
    "high_volatility",
]

volume_event_features = [
    "high_demand",
    "low_demand",
    "high_generation",
    "low_generation",
    "strong_buy_pressure",
    "strong_sell_pressure",
    "buy_volume_spike",
    "sell_volume_spike",
]

combined_event_features = [
    "generation_surplus",
    "demand_pressure",
    "strong_demand_pressure",
    "strong_generation_pressure",
    "demand_driven_price_spike",
    "generation_driven_low_price",
    "scarcity_price_event",
    "oversupply_price_event",
    "buy_pressure_price_spike",
    "sell_pressure_low_price",
    "price_separation",
    "extreme_price_separation",
    "zone_price_outlier",
    "has_combined_event",
]

feature_sets = {
    # Baseline without event variables
    "baseline_price_lags": (
        price_base_features
        + lag_features
    ),

    # Baseline plus volume values, but no event flags
    "baseline_price_lags_volume": (
        price_base_features
        + lag_features
        + volume_base_features
    ),

    # Add only price events
    "price_events": (
        price_base_features
        + lag_features
        + volume_base_features
        + price_event_features
    ),

    # Add price and volume events
    "price_volume_events": (
        price_base_features
        + lag_features
        + volume_base_features
        + price_event_features
        + volume_event_features
    ),

    # Add combined events too
    "all_events": (
        price_base_features
        + lag_features
        + volume_base_features
        + price_event_features
        + volume_event_features
        + combined_event_features
    ),
}

df_model = df_events.dropna()

y = df_model["target"]

print("df_model shape:", df_model.shape)
print("y shape:", y.shape)

for set_name, cols in feature_sets.items():
    missing_cols = [col for col in cols if col not in df_model.columns]
    print(set_name, "n_features:", len(cols), "missing:", missing_cols)

df_model shape: (8736, 58)
y shape: (8736,)
baseline_price_lags n_features: 8 missing: []
baseline_price_lags_volume n_features: 10 missing: []
price_events n_features: 18 missing: []
price_volume_events n_features: 26 missing: []
all_events n_features: 40 missing: []


In [7]:
# ============================================================
# EVENT COUNTS AND PERCENTAGE OF TOTAL DATA
# ============================================================

event_groups = {
    "Price events": price_event_features,
    "Volume events": volume_event_features,
    "Combined events": combined_event_features,
}

total_rows = len(df_model)

event_summary_list = []

for group_name, event_cols in event_groups.items():
    for col in event_cols:
        if col in df_model.columns:
            event_count = df_model[col].sum()
            event_percentage = (event_count / total_rows) * 100

            event_summary_list.append({
                "event_group": group_name,
                "event": col,
                "count": int(event_count),
                "percentage_of_total": event_percentage
            })

event_summary = pd.DataFrame(event_summary_list)

event_summary = event_summary.sort_values(
    ["event_group", "count"],
    ascending=[True, False]
)

event_summary

,event_group,event,count,percentage_of_total
29,Combined events,has_combined_event,903,10.336538
25,Combined events,sell_pressure_low_price,388,4.441392
18,Combined events,strong_demand_pressure,271,3.102106
24,Combined events,buy_pressure_price_spike,265,3.033425
20,Combined events,demand_driven_price_spike,179,2.048993
21,Combined events,generation_driven_low_price,46,0.526557
19,Combined events,strong_generation_pressure,2,0.022894
16,Combined events,generation_surplus,0,0.000000
17,Combined events,demand_pressure,0,0.000000
22,Combined events,scarcity_price_event,0,0.000000


In [8]:
# ============================================================
# 7. TEMPORAL TRAIN / TEST SPLIT BY DATE
# ============================================================

train_end = "2020-11-30"
test_start = "2020-12-01"
test_end = "2020-12-08"

train_mask = df_model.index <= train_end
test_mask = (df_model.index >= test_start) & (df_model.index <= test_end)

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Train y:", y_train.shape)
print("Test y:", y_test.shape)
print("Train period:", y_train.index.min(), "to", y_train.index.max())
print("Test period:", y_test.index.min(), "to", y_test.index.max())

Train y: (7993,)
Test y: (169,)
Train period: 2020-01-02 00:00:00 to 2020-11-30 00:00:00
Test period: 2020-12-01 00:00:00 to 2020-12-08 00:00:00


In [9]:
# ============================================================
# 8. TRAIN MODELS FOR EACH FEATURE SET
# ============================================================

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    "SVR": make_pipeline(
        StandardScaler(),
        SVR(kernel="rbf", C=10, epsilon=0.1)
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

experiment_results = []
experiment_predictions = {}
trained_models = {}

for feature_set_name, feature_cols in feature_sets.items():

    X = df_model[feature_cols]

    X_train = X.loc[train_mask]
    X_test = X.loc[test_mask]

    print(f"Training feature set: {feature_set_name}")
    print("X_train:", X_train.shape, "X_test:", X_test.shape)

    for model_name, model in models.items():
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        medae = median_absolute_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        experiment_results.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "n_features": len(feature_cols),
            "MedAE": medae,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
        })

        experiment_predictions[(feature_set_name, model_name)] = y_pred
        trained_models[(feature_set_name, model_name)] = model

print("All experiments trained successfully")

Training feature set: baseline_price_lags
X_train: (7993, 8) X_test: (169, 8)
Training feature set: baseline_price_lags_volume
X_train: (7993, 10) X_test: (169, 10)
Training feature set: price_events
X_train: (7993, 18) X_test: (169, 18)
Training feature set: price_volume_events
X_train: (7993, 26) X_test: (169, 26)
Training feature set: all_events
X_train: (7993, 40) X_test: (169, 40)
All experiments trained successfully


In [10]:
# ============================================================
# 9. EXPERIMENT RESULTS COMPARISON
# ============================================================

experiment_df = pd.DataFrame(experiment_results)

experiment_df = experiment_df.sort_values(
    ["MedAE", "RMSE", "MAE"],
    ascending=True
)

experiment_df

,feature_set,model,n_features,MedAE,MAE,RMSE,R2
4,baseline_price_lags,SVR,8,0.447722,0.799299,1.168872,0.868641
7,baseline_price_lags_volume,Ridge Regression,10,0.463106,0.752255,1.118646,0.879687
6,baseline_price_lags_volume,Linear Regression,10,0.463143,0.752211,1.118587,0.879700
0,baseline_price_lags,Linear Regression,8,0.464460,0.756247,1.123705,0.878596
1,baseline_price_lags,Ridge Regression,8,0.464479,0.756292,1.123765,0.878583
14,price_events,Random Forest,18,0.468650,0.947489,1.521647,0.777385
3,baseline_price_lags,Gradient Boosting,8,0.473067,0.798822,1.261270,0.847052
13,price_events,Ridge Regression,18,0.480638,0.753460,1.082376,0.887362
12,price_events,Linear Regression,18,0.481412,0.753753,1.082640,0.887307
8,baseline_price_lags_volume,Random Forest,10,0.485600,0.929336,1.508800,0.781128


In [12]:
# ============================================================
# 10. BEST MODEL PER FEATURE SET
# ============================================================

best_by_feature_set = (
    experiment_df
    .sort_values(["feature_set", "MedAE", "MAE", "RMSE"])
    .groupby("feature_set")
    .head(1)
    .sort_values(["MedAE", "MAE", "RMSE"])
)

best_by_feature_set

,feature_set,model,n_features,MedAE,MAE,RMSE,R2
4,baseline_price_lags,SVR,8,0.447722,0.799299,1.168872,0.868641
7,baseline_price_lags_volume,Ridge Regression,10,0.463106,0.752255,1.118646,0.879687
14,price_events,Random Forest,18,0.468650,0.947489,1.521647,0.777385
18,price_volume_events,Linear Regression,26,0.493232,0.747324,1.072952,0.889315
24,all_events,Linear Regression,40,0.499478,0.752324,1.072271,0.889456


In [13]:
# ============================================================
# 11. EXTREME PRICE PERFORMANCE - ML MODELS
# ============================================================

def evaluate_model_on_mask(y_true, y_pred, mask):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    mask = pd.Series(mask).reset_index(drop=True)

    y_true_masked = y_true[mask]
    y_pred_masked = y_pred[mask]

    return {
        "n_obs": len(y_true_masked),
        "MedAE": median_absolute_error(y_true_masked, y_pred_masked),
        "MAE": mean_absolute_error(y_true_masked, y_pred_masked),
        "RMSE": np.sqrt(mean_squared_error(y_true_masked, y_pred_masked)),
        "R2": r2_score(y_true_masked, y_pred_masked) if len(y_true_masked) > 1 else np.nan
    }


# Thresholds computed on real test prices
p10 = y_test.quantile(0.10)
p90 = y_test.quantile(0.90)

high_price_mask = y_test > p90
low_price_mask = y_test < p10
normal_price_mask = (y_test >= p10) & (y_test <= p90)


# Select only the best model of each feature set
selected_models = list(
    zip(best_by_feature_set["feature_set"], best_by_feature_set["model"])
)

extreme_ml_results = []

for feature_set_name, model_name in selected_models:
    y_pred = experiment_predictions[(feature_set_name, model_name)]

    for group_name, mask in {
        "High prices > P90": high_price_mask,
        "Low prices < P10": low_price_mask,
        "Normal prices P10-P90": normal_price_mask
    }.items():

        metrics = evaluate_model_on_mask(y_test, y_pred, mask)

        extreme_ml_results.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "group": group_name,
            **metrics
        })

extreme_ml_df = pd.DataFrame(extreme_ml_results)

extreme_ml_df.sort_values(["group", "MedAE", "MAE", "RMSE"])

,feature_set,model,group,n_obs,MedAE,MAE,RMSE,R2
3,baseline_price_lags_volume,Ridge Regression,High prices > P90,17,0.407822,0.580023,0.783570,-4.341159
6,price_events,Random Forest,High prices > P90,17,0.655200,0.798412,1.036552,-8.346783
12,all_events,Linear Regression,High prices > P90,17,0.665226,0.723951,0.884417,-5.804471
0,baseline_price_lags,SVR,High prices > P90,17,0.673163,0.771663,0.918630,-6.341098
9,price_volume_events,Linear Regression,High prices > P90,17,0.674940,0.720954,0.885435,-5.820148
13,all_events,Linear Regression,Low prices < P10,17,0.263742,0.561905,0.840509,0.678388
7,price_events,Random Forest,Low prices < P10,17,0.295150,0.509518,0.794825,0.712399
10,price_volume_events,Linear Regression,Low prices < P10,17,0.301909,0.561671,0.832218,0.684702
4,baseline_price_lags_volume,Ridge Regression,Low prices < P10,17,0.405204,0.821454,1.225405,0.316394
1,baseline_price_lags,SVR,Low prices < P10,17,0.559919,0.966721,1.313214,0.214913


In [14]:
# ============================================================
# 12. SUDDEN PRICE CHANGE PERFORMANCE - ML MODELS
# ============================================================

# Absolute hourly price change in the test set
test_price_delta = y_test.diff().abs()

# Threshold for strong changes, using P90 of absolute changes
delta_p90 = test_price_delta.quantile(0.90)

strong_change_mask = test_price_delta > delta_p90
normal_change_mask = test_price_delta <= delta_p90

change_ml_results = []

for feature_set_name, model_name in selected_models:
    y_pred = experiment_predictions[(feature_set_name, model_name)]

    for group_name, mask in {
        "Strong changes |delta| > P90": strong_change_mask,
        "Normal changes |delta| <= P90": normal_change_mask
    }.items():

        metrics = evaluate_model_on_mask(y_test, y_pred, mask)

        change_ml_results.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "group": group_name,
            **metrics
        })

change_ml_df = pd.DataFrame(change_ml_results)

change_ml_df.sort_values(["group", "MedAE", "MAE", "RMSE"])

,feature_set,model,group,n_obs,MedAE,MAE,RMSE,R2
3,baseline_price_lags_volume,Ridge Regression,Normal changes |delta| <= P90,151,0.367675,0.539667,0.727808,0.946607
7,price_volume_events,Linear Regression,Normal changes |delta| <= P90,151,0.396393,0.572531,0.776624,0.939204
9,all_events,Linear Regression,Normal changes |delta| <= P90,151,0.397874,0.577856,0.776525,0.939220
1,baseline_price_lags,SVR,Normal changes |delta| <= P90,151,0.418576,0.575578,0.752635,0.942902
5,price_events,Random Forest,Normal changes |delta| <= P90,151,0.446950,0.818981,1.408924,0.799909
4,price_events,Random Forest,Strong changes |delta| > P90,17,2.070900,2.136415,2.320489,0.471450
8,all_events,Linear Regression,Strong changes |delta| > P90,17,2.122981,2.307929,2.459481,0.406236
6,price_volume_events,Linear Regression,Strong changes |delta| > P90,17,2.158394,2.304273,2.461821,0.405106
2,baseline_price_lags_volume,Ridge Regression,Strong changes |delta| > P90,17,2.520596,2.682070,2.781174,0.240753
0,baseline_price_lags,SVR,Strong changes |delta| > P90,17,2.669219,2.807373,2.922185,0.161811


In [15]:
# ============================================================
# 13. DIRECTIONAL ACCURACY - ML MODELS
# ============================================================

def directional_accuracy(y_true, y_pred, mask=None):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)

    true_delta = y_true.diff()
    pred_delta = y_pred.diff()

    valid_mask = true_delta.notna() & pred_delta.notna()

    if mask is not None:
        mask = pd.Series(mask).reset_index(drop=True)
        valid_mask = valid_mask & mask

    true_direction = np.sign(true_delta[valid_mask])
    pred_direction = np.sign(pred_delta[valid_mask])

    return (true_direction == pred_direction).mean()


direction_ml_results = []

for feature_set_name, model_name in selected_models:
    y_pred = experiment_predictions[(feature_set_name, model_name)]

    direction_ml_results.append({
        "feature_set": feature_set_name,
        "model": model_name,
        "Directional_Accuracy_Global": directional_accuracy(y_test, y_pred),
        "Directional_Accuracy_StrongChanges": directional_accuracy(y_test, y_pred, strong_change_mask),
        "Directional_Accuracy_NormalChanges": directional_accuracy(y_test, y_pred, normal_change_mask)
    })

direction_ml_df = pd.DataFrame(direction_ml_results)

direction_ml_df.sort_values(
    ["Directional_Accuracy_StrongChanges", "Directional_Accuracy_Global"],
    ascending=False
)

,feature_set,model,Directional_Accuracy_Global,Directional_Accuracy_StrongChanges,Directional_Accuracy_NormalChanges
2,price_events,Random Forest,0.678571,0.941176,0.649007
0,baseline_price_lags,SVR,0.714286,0.882353,0.695364
1,baseline_price_lags_volume,Ridge Regression,0.690476,0.882353,0.668874
3,price_volume_events,Linear Regression,0.648810,0.882353,0.622517
4,all_events,Linear Regression,0.642857,0.882353,0.615894


In [16]:
# ============================================================
# 14. PRICE CHANGE MAGNITUDE ERROR - ML MODELS
# ============================================================

def evaluate_delta_error(y_true, y_pred, mask=None):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)

    true_delta = y_true.diff()
    pred_delta = y_pred.diff()

    valid_mask = true_delta.notna() & pred_delta.notna()

    if mask is not None:
        mask = pd.Series(mask).reset_index(drop=True)
        valid_mask = valid_mask & mask

    delta_error = true_delta[valid_mask] - pred_delta[valid_mask]

    return {
        "n_obs": valid_mask.sum(),
        "Delta_MedAE": np.median(np.abs(delta_error)),
        "Delta_MAE": np.mean(np.abs(delta_error)),
        "Delta_RMSE": np.sqrt(np.mean(delta_error ** 2)),
        "Mean_real_abs_delta": np.mean(np.abs(true_delta[valid_mask])),
        "Mean_pred_abs_delta": np.mean(np.abs(pred_delta[valid_mask]))
    }


delta_ml_results = []

for feature_set_name, model_name in selected_models:
    y_pred = experiment_predictions[(feature_set_name, model_name)]

    for group_name, mask in {
        "Global delta error": None,
        "Strong changes delta error": strong_change_mask,
        "Normal changes delta error": normal_change_mask
    }.items():

        metrics = evaluate_delta_error(y_test, y_pred, mask)

        delta_ml_results.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "group": group_name,
            **metrics
        })

delta_ml_df = pd.DataFrame(delta_ml_results)

delta_ml_df.sort_values(["group", "Delta_MedAE", "Delta_MAE", "Delta_RMSE"])

,feature_set,model,group,n_obs,Delta_MedAE,Delta_MAE,Delta_RMSE,Mean_real_abs_delta,Mean_pred_abs_delta
3,baseline_price_lags_volume,Ridge Regression,Global delta error,168,0.514647,0.737864,1.055468,0.764524,0.745556
0,baseline_price_lags,SVR,Global delta error,168,0.559478,0.739628,1.057389,0.764524,0.743468
6,price_events,Random Forest,Global delta error,168,0.638075,1.189391,1.913402,0.764524,1.299930
12,all_events,Linear Regression,Global delta error,168,0.693673,0.970725,1.344411,0.764524,0.992876
9,price_volume_events,Linear Regression,Global delta error,168,0.695630,0.975937,1.348442,0.764524,1.000912
2,baseline_price_lags,SVR,Normal changes delta error,151,0.430908,0.648033,0.937115,0.537815,0.599426
5,baseline_price_lags_volume,Ridge Regression,Normal changes delta error,151,0.434691,0.636527,0.919049,0.537815,0.607281
8,price_events,Random Forest,Normal changes delta error,151,0.600850,1.162827,1.934998,0.537815,1.190833
14,all_events,Linear Regression,Normal changes delta error,151,0.645062,0.914919,1.279430,0.537815,0.858939
11,price_volume_events,Linear Regression,Normal changes delta error,151,0.658640,0.921827,1.284990,0.537815,0.866738
